In [ ]:

import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from nilearn import datasets, plotting
from nilearn.image import mean_img
from nilearn.maskers import NiftiLabelsMasker


In [ ]:

# --------------------------------------------------
# 1. Locate preprocessed functional images
# --------------------------------------------------

CLEAN_DIR = Path.home() / "nilearn_data"

cleaned_files = sorted(
    CLEAN_DIR.glob("*_cleaned.nii.gz")
)

if not cleaned_files:
    raise FileNotFoundError(
        f"No cleaned images found in: {CLEAN_DIR}"
    )

print("Number of cleaned subjects:", len(cleaned_files))

# --------------------------------------------------
# 2. Fetch Schaefer atlas
# --------------------------------------------------

atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100,
    yeo_networks=7,
    resolution_mm=2
)

print("Atlas labels:", len(atlas.labels))

# --------------------------------------------------
# 3. Initialize masker
# --------------------------------------------------

# The input images have already been preprocessed.
# Do not repeat confound regression, filtering,
# detrending, or volume censoring.

masker = NiftiLabelsMasker(
    labels_img=atlas.maps,
    standardize=None,
    detrend=False,
    high_pass=None,
    low_pass=None,
    resampling_target="data",
    memory="nilearn_cache",
    verbose=0
)

# --------------------------------------------------
# 4. Process each subject
# --------------------------------------------------

RESULTS_DIR = CLEAN_DIR.parent / "fc_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for cleaned_file in cleaned_files:

    # 1. Subject ID
    sid = cleaned_file.name.replace("_cleaned.nii.gz", "")

    print(f"\nProcessing: {sid}")

    # 2. Load preprocessed image
    img = nib.load(str(cleaned_file))

    print("Preprocessed image shape:", img.shape)

    # 3. Extract ROI time series
    ts = masker.fit_transform(str(cleaned_file))

    print("ROI time series shape:", ts.shape)

    # 4. Calculate Pearson correlation
    fc = np.corrcoef(ts, rowvar=False)

    print("FC matrix shape:", fc.shape)

    # 5. Calculate Fisher Z
    fc_clipped = np.clip(fc, -1 + 1e-7, 1 - 1e-7)
    fc_z = np.arctanh(fc_clipped)
    np.fill_diagonal(fc_z, np.nan)

    # 6. Save outputs
    np.save(RESULTS_DIR / f"{sid}_timeseries.npy", ts)
    np.save(RESULTS_DIR / f"{sid}_pearson.npy", fc)
    np.save(RESULTS_DIR / f"{sid}_fisher_z.npy", fc_z)

    print(f"Subject {sid} completed.")

print("\nAll subjects completed.")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets

RESULTS_DIR = Path.home() / "fc_results"


In [ ]:

CLEAN_DIR = Path.home() / "nilearn_data"

cleaned_files = sorted(
    CLEAN_DIR.glob("*_cleaned.nii.gz")
)
RESULTS_DIR = CLEAN_DIR.parent / "fc_results"

# --------------------------------------------------
# 1.Netwrok extraction 
# --------------------------------------------------
atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100, yeo_networks=7, resolution_mm=2
)

labels = [l.decode() if isinstance(l, bytes) else str(l) for l in atlas.labels]
labels = [l for l in labels if l.lower() != "background"]   

assert len(labels) == 100, f"Unexpected number of labels: {len(labels)}"


roi_network = np.array([l.split("_")[2] for l in labels])

# Yeo-7
network_order = ["Vis", "SomMot", "DorsAttn", "SalVentAttn",
                 "Limbic", "Cont", "Default"]
assert set(roi_network) == set(network_order), set(roi_network)

net_idx = {net: np.where(roi_network == net)[0] for net in network_order}

print({net: len(idx) for net, idx in net_idx.items()})   

# --------------------------------------------------
# 2. Network to network matrix
# --------------------------------------------------
def network_fc(fc_z, net_idx, network_order):
    n = len(network_order)
    out = np.full((n, n), np.nan)

    for i, a in enumerate(network_order):
        for j, b in enumerate(network_order):
            idx_a, idx_b = net_idx[a], net_idx[b]
            block = fc_z[np.ix_(idx_a, idx_b)]

            if i == j:
                # درون‌شبکه‌ای: فقط مثلث بالا، بدون قطر
                vals = block[np.triu_indices(len(idx_a), k=1)]
            else:
                vals = block.ravel()

            out[i, j] = np.nanmean(vals)
    return out

# --------------------------------------------------
# 3. Loop on all
# --------------------------------------------------
fz_files = sorted(RESULTS_DIR.glob("*_fisher_z.npy"))
print("Number of subjects:", len(fz_files))

subject_ids, net_mats = [], []

for f in fz_files:
    sid = f.name.replace("_fisher_z.npy", "")
    fc_z = np.load(f)
    assert fc_z.shape == (100, 100), (sid, fc_z.shape)

    nm = network_fc(fc_z, net_idx, network_order)
    np.save(RESULTS_DIR / f"{sid}_network_fc.npy", nm)

    subject_ids.append(sid)
    net_mats.append(nm)

net_mats = np.stack(net_mats)          # (n_subjects, 7, 7)
np.save(RESULTS_DIR / "all_network_fc.npy", net_mats)

# --------------------------------------------------
# 4. statistics 
# --------------------------------------------------
rows = []
for s, sid in enumerate(subject_ids):
    for i, a in enumerate(network_order):
        for j, b in enumerate(network_order):
            if j < i:      # فقط مثلث بالا (ماتریس متقارن است)
                continue
            rows.append({
                "subject": sid,
                "network_1": a,
                "network_2": b,
                "type": "within" if i == j else "between",
                "fisher_z": net_mats[s, i, j],
            })

net_df = pd.DataFrame(rows)
net_df.to_csv(RESULTS_DIR / "network_fc_long.csv", index=False)

# --------------------------------------------------
# 5. group mean
# --------------------------------------------------
group_mean = np.nanmean(net_mats, axis=0)

fig, ax = plt.subplots(figsize=(7, 6))
vmax = np.nanmax(np.abs(group_mean))
im = ax.imshow(group_mean, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(7)); ax.set_xticklabels(network_order, rotation=45, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(network_order)

for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{group_mean[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, label="Mean Fisher z")
ax.set_title("Group-mean network-level FC (Schaefer-100, Yeo-7)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "network_fc_group_mean.png", dpi=300)
plt.show()

In [ ]:



# ==================================================
# 1. PATHS AND SETTINGS
# ==================================================

CLEAN_DIR = Path.home() / "nilearn_data"

RESULTS_DIR = Path.home() / "fc_results"

OUT_DIR = RESULTS_DIR / "brain_visualizations"

OUT_DIR.mkdir(parents=True, exist_ok=True)

EDGE_THRESHOLD = "97%"  # Show strongest 3% of edges


NETWORK_ORDER = [
    "Vis",
    "SomMot",
    "DorsAttn",
    "SalVentAttn",
    "Limbic",
    "Cont",
    "Default"
]


NETWORK_COLORS = {
    "Vis": "#D55E00",
    "SomMot": "#0072B2",
    "DorsAttn": "#009E73",
    "SalVentAttn": "#CC79A7",
    "Limbic": "#E69F00",
    "Cont": "#56B4E9",
    "Default": "#6A3D9A"
}


# ==================================================
# 2. IDENTIFY THE FIVE REAL SUBJECTS
# ==================================================

cleaned_files = sorted(
    CLEAN_DIR.glob("*_cleaned.nii.gz")
)

subject_ids = [
    f.name.replace("_cleaned.nii.gz", "")
    for f in cleaned_files
]

if len(subject_ids) != 5:
    raise ValueError(
        f"Expected 5 subjects, found {len(subject_ids)}"
    )

print("Subjects:", subject_ids)


# ==================================================
# 3. FETCH SCHAEFER-100 ATLAS
# ==================================================

atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100,
    yeo_networks=7,
    resolution_mm=2
)

atlas_img = nib.load(str(atlas.maps))

atlas_data = np.rint(
    np.asarray(atlas_img.dataobj)
).astype(int)

roi_ids = np.sort(
    np.unique(atlas_data[atlas_data > 0])
)

assert np.array_equal(
    roi_ids,
    np.arange(1, 101)
)


# ==================================================
# 4. EXTRACT ROI LABELS AND NETWORK MEMBERSHIP
# ==================================================

roi_labels = [
    label.decode() if isinstance(label, bytes)
    else str(label)
    for label in atlas.labels
]

roi_labels = [
    label for label in roi_labels
    if label.lower() != "background"
]

assert len(roi_labels) == 100

roi_network = np.array([
    label.split("_")[2]
    for label in roi_labels
])

assert set(roi_network) == set(NETWORK_ORDER)


# ==================================================
# 5. EXTRACT MNI COORDINATES OF 100 ROIs
# ==================================================

coords = []

for roi_id in roi_ids:

    voxels = np.argwhere(
        atlas_data == roi_id
    )

    ijk_center = voxels.mean(axis=0)

    xyz_center = nib.affines.apply_affine(
        atlas_img.affine,
        ijk_center
    )

    coords.append(xyz_center)

coords = np.asarray(coords)

assert coords.shape == (100, 3)


# ==================================================
# 6. ASSIGN COLORS TO 100 ROI NODES
# ==================================================

roi_colors = [
    NETWORK_COLORS[network]
    for network in roi_network
]


# ==================================================
# 7. LOAD INDIVIDUAL FISHER Z MATRICES
# ==================================================

z_matrices = []

off_diagonal = ~np.eye(100, dtype=bool)

for sid in subject_ids:

    file = RESULTS_DIR / f"{sid}_fisher_z.npy"

    if not file.exists():
        raise FileNotFoundError(file)

    z = np.load(file)

    assert z.shape == (100, 100)

    if not np.isfinite(z[off_diagonal]).all():
        raise ValueError(
            f"Invalid values in {sid}"
        )

    if not np.allclose(
        z,
        z.T,
        atol=1e-6,
        equal_nan=True
    ):
        raise ValueError(
            f"Asymmetric FC matrix: {sid}"
        )

    z = z.copy()

    # Remove diagonal self-connections
    np.fill_diagonal(z, 0)

    z_matrices.append(z)


# ==================================================
# 8. GROUP MEAN FISHER Z
# ==================================================

z_stack = np.stack(z_matrices)

group_mean_z = np.mean(
    z_stack,
    axis=0
)

# Return to Pearson correlation scale
group_mean_r = np.tanh(group_mean_z)

np.fill_diagonal(group_mean_r, 0)


np.save(
    OUT_DIR / "group_mean_roi_fc_r.npy",
    group_mean_r
)


# ==================================================
# 9. STATIC BRAIN CONNECTOME
# ==================================================

fig = plt.figure(figsize=(14, 8))

display = plotting.plot_connectome(

    adjacency_matrix=group_mean_r,

    node_coords=coords,

    node_color=roi_colors,

    node_size=28,

    edge_cmap="RdBu_r",

    edge_vmin=-1,

    edge_vmax=1,

    edge_threshold=EDGE_THRESHOLD,

    display_mode="lyrz",

    figure=fig,

    axes=(0.02, 0.15, 0.96, 0.80),

    title="Group Mean Functional Connectome (Schaefer-100)",

    colorbar=True,

    annotate=False

)


# Network legend
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=NETWORK_COLORS[net],
        markeredgecolor="black",
        markersize=9,
        label=net
    )
    for net in NETWORK_ORDER
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=False
)

fig.savefig(
    OUT_DIR / "group_roi_connectome.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ==================================================
# 10. INTERACTIVE 3D CONNECTOME
# ==================================================

view = plotting.view_connectome(

    adjacency_matrix=group_mean_r,

    node_coords=coords,

    node_color=roi_colors,

    node_size=5,

    edge_threshold=EDGE_THRESHOLD,

    edge_cmap="RdBu_r",

    symmetric_cmap=True,

    title="Interactive Group Functional Connectome"

)

view.save_as_html(
    str(OUT_DIR / "group_roi_connectome_3D.html")
)


print("\nROI-level connectome completed.")
print("Output directory:", OUT_DIR)

In [ ]:

from packaging.version import Version
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from nilearn import datasets, plotting, surface


# ============================================================
# 1. SETTINGS
# ============================================================

OUT_DIR = (
    Path.home()
    / "fc_results"
    / "network_surface_maps"
)

INDIVIDUAL_DIR = OUT_DIR / "individual_networks"

OUT_DIR.mkdir(parents=True, exist_ok=True)
INDIVIDUAL_DIR.mkdir(parents=True, exist_ok=True)


NETWORK_ORDER = [
    "Vis",
    "SomMot",
    "DorsAttn",
    "SalVentAttn",
    "Limbic",
    "Cont",
    "Default"
]

NETWORK_COLORS = {
    "Vis": "#E64B35",
    "SomMot": "#4DBBD5",
    "DorsAttn": "#00A087",
    "SalVentAttn": "#7E6148",
    "Limbic": "#EFC000",
    "Cont": "#3C5488",
    "Default": "#9B59B6"
}

# Grey represents vertices outside the assigned networks
ALL_NETWORK_COLORS = [
    "#D9D9D9"
] + [
    NETWORK_COLORS[net]
    for net in NETWORK_ORDER
]

NETWORK_CMAP = ListedColormap(ALL_NETWORK_COLORS)


print("Nilearn version:", nilearn.__version__)


# ============================================================
# 2. FETCH SCHAEFER-100 ATLAS
# ============================================================

atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100,
    yeo_networks=7,
    resolution_mm=2
)

atlas_img = nib.load(str(atlas.maps))

atlas_data = np.rint(
    np.asarray(atlas_img.dataobj)
).astype(np.int16)

print("Atlas shape:", atlas_data.shape)


# ============================================================
# 3. MAP EACH ROI TO ITS YEO-7 NETWORK
# ============================================================

raw_labels = [
    label.decode() if isinstance(label, bytes)
    else str(label)
    for label in atlas.labels
]

# IMPORTANT:
# Atlas voxel values correspond to indices in atlas.labels.
# Therefore, preserve the original label indices when
# excluding the background label.

roi_to_network = {}

for roi_id, label in enumerate(raw_labels):

    if label.lower() == "background":
        continue

    parts = label.split("_")

    if len(parts) < 4:
        raise ValueError(f"Unexpected atlas label: {label}")

    network = parts[2]

    if network not in NETWORK_ORDER:
        raise ValueError(f"Unknown network: {network}")

    roi_to_network[roi_id] = network


present_roi_ids = set(
    np.unique(atlas_data[atlas_data > 0]).tolist()
)

missing_ids = present_roi_ids - set(roi_to_network)

if missing_ids:
    raise ValueError(
        f"Some atlas ROI IDs have no network labels: {missing_ids}"
    )

if len(present_roi_ids) != 100:
    raise ValueError(
        f"Expected 100 ROIs, found {len(present_roi_ids)}"
    )

print("\nROIs per network:")

for network in NETWORK_ORDER:

    count = sum(
        net == network
        for net in roi_to_network.values()
    )

    print(f"{network}: {count}")


# ============================================================
# 4. CREATE A VOLUMETRIC YEO-7 NETWORK MAP
# ============================================================

# 0 = background
# 1 = Vis
# 2 = SomMot
# ...
# 7 = Default

network_volume = np.zeros(
    atlas_data.shape,
    dtype=np.int16
)

for roi_id, network in roi_to_network.items():

    network_code = NETWORK_ORDER.index(network) + 1

    network_volume[atlas_data == roi_id] = network_code


assert set(
    np.unique(network_volume)
) == set(range(8))

network_img = nib.Nifti1Image(
    network_volume,
    atlas_img.affine,
    atlas_img.header.copy()
)


# ============================================================
# 5. FETCH FSAVERAGE-5 SURFACE
# ============================================================

# fsaverage5 has 10,242 vertices per hemisphere.
# It is a relatively lightweight option for visualization.

fsaverage = datasets.fetch_surf_fsaverage(
    mesh="fsaverage5"
)


# ============================================================
# 6. PROJECT NETWORK LABELS FROM VOLUME TO CORTICAL SURFACE
# ============================================================

def project_network_to_surface(hemi):

    """
    Project categorical Yeo-7 network labels
    onto one cortical hemisphere.

    Pial surface = projection surface
    White surface = inner cortical surface

    Nearest-most-frequent interpolation preserves
    discrete atlas labels.
    """

    pial_mesh = fsaverage[f"pial_{hemi}"]

    white_mesh = fsaverage[f"white_{hemi}"]

    if Version(nilearn.__version__) >= Version("0.12.0"):

        interpolation = "nearest_most_frequent"
        depths = [0.2, 0.5, 0.8]

    else:

        # Compatibility with older Nilearn versions.
        interpolation = "nearest"
        depths = [0.5]

    surf_data = surface.vol_to_surf(

        img=network_img,

        surf_mesh=pial_mesh,

        inner_mesh=white_mesh,

        kind="depth",

        depth=depths,

        interpolation=interpolation

    )

    # Replace undefined values with background.
    surf_data = np.nan_to_num(
        surf_data,
        nan=0
    )

    # Restore discrete integer network labels.
    surf_data = np.rint(
        surf_data
    ).astype(np.int16)

    if not np.isin(
        surf_data,
        np.arange(8)
    ).all():

        raise ValueError(
            f"Unexpected network values in {hemi} hemisphere"
        )

    return surf_data


print("\nProjecting networks onto cortical surfaces...")

surface_labels = {

    "left": project_network_to_surface("left"),

    "right": project_network_to_surface("right")

}

for hemi, data in surface_labels.items():

    print(
        f"{hemi.capitalize()} hemisphere:",
        len(data),
        "vertices"
    )

    print(
        "Assigned network vertices:",
        np.sum(data > 0)
    )


# Save projected labels for reproducibility.
np.savez_compressed(
    OUT_DIR / "yeo7_surface_labels.npz",
    left=surface_labels["left"],
    right=surface_labels["right"]
)


# ============================================================
# 7. DEFINE FOUR CORTICAL VIEWS
# ============================================================

VIEWS = [

    ("left", "lateral", "Left lateral"),

    ("right", "lateral", "Right lateral"),

    ("left", "medial", "Left medial"),

    ("right", "medial", "Right medial")

]


# ============================================================
# 8. PLOT ALL SEVEN NETWORKS TOGETHER
# ============================================================

fig = plt.figure(figsize=(16, 12))

fig.suptitle(
    "Schaefer-100 | Yeo-7 Cortical Networks",
    fontsize=19,
    fontweight="bold",
    y=0.97
)

for i, (hemi, view, title) in enumerate(VIEWS):

    ax = fig.add_subplot(
        2,
        2,
        i + 1,
        projection="3d"
    )

    plotting.plot_surf_roi(

        surf_mesh=fsaverage[f"infl_{hemi}"],

        roi_map=surface_labels[hemi],

        bg_map=fsaverage[f"sulc_{hemi}"],

        hemi=hemi,

        view=view,

        cmap=NETWORK_CMAP,

        vmin=0,

        vmax=7,

        colorbar=False,

        bg_on_data=True,

        axes=ax,

        title=title

    )


# Create network legend.
legend_handles = [

    Patch(
        facecolor=NETWORK_COLORS[net],
        edgecolor="black",
        label=net
    )

    for net in NETWORK_ORDER

]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=11
)

plt.tight_layout(rect=[0, 0.08, 1, 0.94])

plt.savefig(
    OUT_DIR / "all_seven_networks_surface.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)


# ============================================================
# 9. PLOT EACH NETWORK SEPARATELY
# ============================================================

for network in NETWORK_ORDER:

    network_code = NETWORK_ORDER.index(network) + 1

    network_color = NETWORK_COLORS[network]

    # Make binary surface maps for the selected network.
    # 1 = selected network
    # 0 = other cortical regions

    network_cmap = ListedColormap([
        "#D9D9D9",
        network_color
    ])

    fig = plt.figure(figsize=(15, 11))

    fig.suptitle(
        f"{network} Network | Schaefer-100",
        fontsize=19,
        fontweight="bold",
        y=0.97
    )

    for i, (hemi, view, title) in enumerate(VIEWS):

        # Retain only the selected network.
        network_surface = (
            surface_labels[hemi] == network_code
        ).astype(np.int16)

        ax = fig.add_subplot(
            2,
            2,
            i + 1,
            projection="3d"
        )

        plotting.plot_surf_roi(

            surf_mesh=fsaverage[f"infl_{hemi}"],

            roi_map=network_surface,

            bg_map=fsaverage[f"sulc_{hemi}"],

            hemi=hemi,

            view=view,

            cmap=network_cmap,

            vmin=0,

            vmax=1,

            colorbar=False,

            bg_on_data=True,

            axes=ax,

            title=title

        )

    plt.tight_layout(rect=[0, 0.02, 1, 0.94])

    output_file = (
        INDIVIDUAL_DIR
        / f"{network}_surface.png"
    )

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print("Saved:", output_file)


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)

print("YEO-7 CORTICAL NETWORK VISUALIZATION COMPLETED")

print("=" * 60)

print("Main figure:")
print(
    OUT_DIR / "all_seven_networks_surface.png"
)

print("\nIndividual network figures:")

for network in NETWORK_ORDER:

    print(
        INDIVIDUAL_DIR / f"{network}_surface.png"
    )

print("\nProjected surface labels:")
print(
    OUT_DIR / "yeo7_surface_labels.npz"
)

print("\nOutput directory:")
print(OUT_DIR)
